## What Middleware Is
### Definition

Middleware is a layer that runs before and/or after your route handler.

It sits between the client request and your application logic.

## pipeline
Think of it as a processing pipeline:

Client → Middleware → Routing → Dependencies → Endpoint

Endpoint → Middleware → Client

Middleware can:

- Inspect requests

- Modify requests

- Block requests

- Measure performance

- Modify responses

- Add headers

It runs for every request, unless conditionally applied.

## Built-in Middlewares in FastAPI
FastAPI uses Starlette middleware internally.

### . CORS Middleware

CORS = Cross-Origin Resource Sharing.

#### Used when:

Frontend (React, Next.js, etc.) runs on different domain than backend.

Without CORS → browser blocks requests

In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # allow all domain like (React, Next.js, etc)
    allow_methods=["*"], # allow all method like get , post, put , patch
    allow_headers=["*"], # allow all request response header

)
print()

### Trusted Host Middleware

Protects against Host header attacks.

In [ ]:
from fastapi.middleware.trustedhost import TrustedHostMiddleware

app.add_middleware(
    TrustedHostMiddleware,
    allowed_hosts=["example.com", "*.example.com"],
)

# FastAPI Built-in Middleware Reference

| Middleware | Purpose | Example Usage |
|------------|---------|---------------|
| **CORSMiddleware** | Handles Cross-Origin Resource Sharing so browsers from other domains can access your API | ```python\nfrom fastapi.middleware.cors import CORSMiddleware\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=[\"*\"],\n    allow_methods=[\"*\"],\n    allow_headers=[\"*\"],\n)\n``` |
| **GZipMiddleware** | Compresses responses using GZip to reduce payload size | ```python\nfrom fastapi.middleware.gzip import GZipMiddleware\napp.add_middleware(GZipMiddleware, minimum_size=1000)\n``` |
| **TrustedHostMiddleware** | Ensures requests only come from allowed hostnames | ```python\nfrom starlette.middleware.trustedhost import TrustedHostMiddleware\napp.add_middleware(TrustedHostMiddleware, allowed_hosts=[\"example.com\", \"*.example.com\"])\n``` |
| **HTTPSRedirectMiddleware** | Redirects all HTTP requests to HTTPS | ```python\nfrom starlette.middleware.httpsredirect import HTTPSRedirectMiddleware\napp.add_middleware(HTTPSRedirectMiddleware)\n``` |
| **SessionMiddleware** | Adds server-side session management using cookies | ```python\nfrom starlette.middleware.sessions import SessionMiddleware\napp.add_middleware(SessionMiddleware, secret_key=\"your-secret-key\")\n``` |
| **BaseHTTPMiddleware** | Base class for creating custom middleware with request/response handling | ```python\nfrom starlette.middleware.base import BaseHTTPMiddleware\nclass CustomMiddleware(BaseHTTPMiddleware):\n    async def dispatch(self, request, call_next):\n        # custom logic\n        response = await call_next(request)\n        return response\napp.add_middleware(CustomMiddleware)\n``` |

## Writing Custom Middleware

In [ ]:
from fastapi import Request
import time

@app.middleware("http")
async def custom_middleware(request: Request, call_next):
    start_time = time.time()

    response = await call_next(request)

    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)

    return response

**request** → incoming request object

**call_next(request)** → passes control to next layer (routing, dependencies, endpoint)

Code before call_next = runs before endpoint

Code after call_next = runs after endpoint

### some custom middelware
- Logging Middleware
- Request ID Middleware
- Timing Middleware
- 

# Custom Middleware Types in FastAPI

| Middleware Type | Purpose | FastAPI Example Snippet |
|-----------------|---------|------------------------|
| **Logging** | Logs requests and responses for debugging or auditing | ```python\n@app.middleware("http")\nasync def logging_middleware(request, call_next):\n    print(f"Request: {request.method} {request.url}")\n    response = await call_next(request)\n    print(f"Response status: {response.status_code}")\n    return response\n``` |
| **Authentication** | Validates JWT or API key before processing | ```python\n@app.middleware("http")\nasync def auth_middleware(request, call_next):\n    token = request.headers.get("Authorization")\n    if token != "SECRET":\n        return JSONResponse({"error": "Unauthorized"}, status_code=401)\n    return await call_next(request)\n``` |
| **Rate Limiting** | Restricts requests per client to prevent abuse | ```python\nfrom starlette.responses import JSONResponse\ncounter = {}\n@app.middleware("http")\nasync def rate_limit(request, call_next):\n    ip = request.client.host\n    counter[ip] = counter.get(ip, 0) + 1\n    if counter[ip] > 5:\n        return JSONResponse({'error': 'Too many requests'}, status_code=429)\n    return await call_next(request)\n``` |
| **Request ID** | Generates unique ID per request for logging/tracking | ```python\nimport uuid\n@app.middleware("http")\nasync def request_id_middleware(request, call_next):\n    request.state.id = str(uuid.uuid4())\n    response = await call_next(request)\n    response.headers["X-Request-ID"] = request.state.id\n    return response\n``` |
| **Error Handling** | Catches exceptions and returns formatted response | ```python\n@app.middleware("http")\nasync def error_middleware(request, call_next):\n    try:\n        return await call_next(request)\n    except Exception as e:\n        return JSONResponse({'error': str(e)}, status_code=500)\n``` |
| **Input Validation / Sanitization** | Validates or cleans request data | ```python\n@app.middleware("http")\nasync def sanitize_middleware(request, call_next):\n    body = await request.json()\n    if not body:\n        return JSONResponse({'error':'Empty request body'}, status_code=400)\n    return await call_next(request)\n``` |
| **Response Transformation** | Modifies responses before sending | ```python\n@app.middleware("http")\nasync def add_metadata_middleware(request, call_next):\n    response = await call_next(request)\n    if response.media_type == "application/json":\n        data = await response.json()\n        data['server_time'] = str(datetime.utcnow())\n        return JSONResponse(data)\n    return response\n``` |
| **Feature Toggle / AB Testing** | Enables/disables features per user | ```python\n@app.middleware("http")\nasync def feature_toggle(request, call_next):\n    request.state.new_ui = request.headers.get('X-New-UI') == '1'\n    return await call_next(request)\n``` |